## Reading and Preparing a Research PDF for RAG using LangChain

In [ ]:
#%run /Users/csharm33/code/genai_dbx/Course3/.setup/learner_setup.ipynb

In [ ]:
# Import required libraries
import os
import nltk
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from nltk.tokenize import sent_tokenize

In [ ]:
nltk.data.path.append("/Users/csharm33/code/genai_dbx/Course3/Module2/Data/nltk/nltk_data")

In [ ]:
# Define a function to load and extract text from PDF
def load_pdf_with_langchain(pdf_path):
    
    # Use LangChain's built-in loader
    loader = PyMuPDFLoader(pdf_path)

    # Load the PDF into LangChain's document format
    documents = loader.load()

    print(f"Successfully loaded {len(documents)} document chunks from the PDF.")
    return documents

In [ ]:
# Path to the uploaded PDF (replace with your actual file path)
pdf_path = "/Users/csharm33/code/genai_dbx/Course3/Module2/Data/HealthcaredocforRAG.pdf" 

# Extract the document chunks
docs = load_pdf_with_langchain(pdf_path)


## Method 1: Fixed-Size Chunking

We break text into chunks based on a fixed number of **characters**. 
Simple and fast — but may cut off sentences halfway.



### What is `chunk_size` and `chunk_overlap`?

- **`chunk_size`**:  
  The number of characters in each chunk.  
  → Example: `chunk_size=500` creates chunks of 500 characters.

- **`chunk_overlap`**:  
 The number of characters repeated between chunks to keep context.  
  → Example: `chunk_overlap=50` means the last 50 characters of one chunk appear at the start of the next.

Use overlap to avoid cutting off sentences or breaking flow across chunks.



## Types of Fixed Chunking

There are two types of **fixed chunking** in LangChain:

1. **CharacterTextSplitter**
2. **RecursiveCharacterTextSplitter**

Both are used to split long documents into smaller parts (chunks), but they work differently.

### CharacterTextSplitter
- Splits text into chunks based on a **fixed number of characters**.
- Uses a **single separator** (e.g. space `" "` or newline `"\n"`).
- If a sentence or paragraph is too long, it may split mid-sentence.
- Simple and fast, but may not always preserve context cleanly.



### RecursiveCharacterTextSplitter
- Tries to split text at **natural boundaries**: paragraphs → sentences → words → characters.
- Uses a **list of separators**, and recursively falls back if cleaner splits aren't possible.
- Produces better structured chunks with less context loss.
- More advanced and preserves semantic meaning better.

> Use `RecursiveCharacterTextSplitter` when you want **smart chunking**, especially for documents with mixed formatting or longer sentences.


In [ ]:
# Define a function to split text into fixed-size character chunks using LangChain's CharacterTextSplitter.

def fixed_size_chunking(docs, chunk_size=500, chunk_overlap=50):
    
    splitter = CharacterTextSplitter(
        separator=" ",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(docs)

# Apply it
fixed_chunks = fixed_size_chunking(docs)
print(f" Total fixed-size chunks: {len(fixed_chunks)}\n")
print(f" Example:First Chunk \n{fixed_chunks[0].page_content[:]}")

In [ ]:
# Splits documents using RecursiveCharacterTextSplitter which preserves context better

def recursive_chunking(docs, chunk_size=500, chunk_overlap=50):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(docs)

# Apply it
recursive_chunks = recursive_chunking(docs)
print(f" Total recursive chunks: {len(recursive_chunks)}\n")
print(f" Example: First Chunk \n{recursive_chunks[0].page_content[:]}")



## Method 2: Sentence-Based Chunking

Here, we split the text into a specified number of **sentences per chunk**. 
More readable and retains meaning better.

### What is `sentences_per_chunk`?

- **`sentences_per_chunk`** defines **how many sentences** to include in each chunk.
  → Example: `sentences_per_chunk=3` will group 3 sentences together as one chunk.

- It keeps the chunks **meaningful and readable** by not cutting through sentences.

Great for preserving logical flow and making sure each chunk conveys a complete thought or mini-topic.



In [ ]:
# Define a function to split each page into chunks of N sentences.
def sentence_based_chunking(docs, sentences_per_chunk=3):

    chunks = []

    for doc in docs:
        sentences = sent_tokenize(doc.page_content)
        for i in range(0, len(sentences), sentences_per_chunk):
            chunk_text = " ".join(sentences[i:i + sentences_per_chunk])
            chunks.append(chunk_text)

    return chunks

sentence_chunks = sentence_based_chunking(docs)
print(f" Total sentence-based chunks: {len(sentence_chunks)}\n")
print(f" Example:\n{sentence_chunks[0][:]}")



## Method 3: Semantic Chunking

Semantic chunking splits content based on **paragraph breaks or logical boundaries**.
This helps preserve the meaning of each idea.

In [ ]:
# Define a function to split based on paragraph breaks (using two newlines)

def semantic_chunking(docs):
   
    chunks = []
    for doc in docs:
        paragraphs = doc.page_content.split("\n\n")
        for para in paragraphs:
            cleaned = para.strip()
            if cleaned:
                chunks.append(cleaned)
    return chunks

semantic_chunks = semantic_chunking(docs)
print(f" Total semantic chunks: {len(semantic_chunks)}")
print(f" Example:\n{semantic_chunks[0][:]}")



#### If we look closely, we’ll see that **the chunk at index 0** contains **the same content across all the methods**—Fixed Chunking, Recursive Chunking, Sentence-Based Chunking, and Semantic Chunking. However, when we **invoke the chunk at index 6**, the **differences among the methods become clear**.


In [ ]:
fixed_chunks = fixed_size_chunking(docs)
#print(f" Total fixed-size chunks : {len(fixed_chunks)}")
print(f" Fixed-size chunk at index 6: \n{fixed_chunks[6].page_content[:]}\n")

recursive_chunks = recursive_chunking(docs)
#print(f" Total recursive chunks: {len(recursive_chunks)}")
print(f" Recursive chunk at index 6: \n{recursive_chunks[6].page_content[:]}\n")

sentence_chunks = sentence_based_chunking(docs)
#print(f" Total sentence-based chunks: {len(sentence_chunks)}")
print(f" Sentence-based chunk at index 6:\n{sentence_chunks[6][:]}\n")

semantic_chunks = semantic_chunking(docs)
#print(f" Total semantic chunks: {len(semantic_chunks)}")
print(f" Semantic chunk at index 6:\n{semantic_chunks[6][:]}")
